# Hybrid Robotic Vision: Notebook 6 — Results Aggregation and Paper Plots

This notebook aggregates the policy-specific hybrid summaries and generates paper-ready tables and plots.

It assumes that Notebook 5 has already produced files like:

```text
runs/
  hybrid_summary/
    <sequence_name>/
      <regime>/
        <policy_name>/
          hybrid_system_summary.json
```

## What this notebook does

1. Scans all available hybrid summaries  
2. Builds one aggregated dataframe  
3. Saves combined CSV tables  
4. Produces paper-ready matplotlib plots for:
   - semantic gain vs ROI bitrate
   - semantic gain vs ROI bitrate share
   - selected ROIs by policy
   - positive confidence gain rate by policy
   - confidence gain per KB by policy
   - entropy gain vs ROI bitrate

## Main outputs

```text
runs/
  paper_plots/
    aggregated_hybrid_summary.csv
    aggregated_policy_summary.csv
    paper_policy_comparison_table.csv
    plot_conf_gain_vs_roi_bitrate.png
    plot_conf_gain_vs_roi_share.png
    plot_selected_rois_<regime>.png
    plot_positive_conf_gain_rate_<regime>.png
    plot_conf_gain_per_kb_<regime>.png
    plot_entropy_gain_vs_roi_bitrate.png
```

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Define paths

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/hybrid_robotic_vision")
RUNS = ROOT / "runs"
SUMMARY_ROOT = RUNS / "hybrid_summary"
OUT_DIR = RUNS / "paper_plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Summary root:", SUMMARY_ROOT)
print("Plot output dir:", OUT_DIR)

## 3. Imports

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. Scan all hybrid summaries

In [ ]:
summary_paths = sorted(SUMMARY_ROOT.glob("*/*/*/hybrid_system_summary.json"))

print(f"Found {len(summary_paths)} summary files")
for p in summary_paths[:20]:
    print(p)

In [ ]:
rows = []

for p in summary_paths:
    with open(p, "r") as f:
        d = json.load(f)
    rows.append(d)

summary_df = pd.DataFrame(rows)
print(summary_df.shape)
summary_df.head()

## 5. Save aggregated CSV

In [ ]:
aggregated_csv = OUT_DIR / "aggregated_hybrid_summary.csv"
summary_df.to_csv(aggregated_csv, index=False)

print("Saved:", aggregated_csv)

## 6. Basic cleanup and ordering

In [ ]:
main_cols = [
    "sequence_name",
    "regime",
    "policy_name",
    "num_raw_detections",
    "num_selected_rois",
    "roi_selection_ratio",
    "frame_roi_coverage",
    "roi_rate_hz",
    "mean_roi_bytes",
    "total_roi_bytes",
    "roi_bitrate_mbps",
    "roi_bitrate_share",
    "video_bitrate_mbps",
    "hybrid_bitrate_mbps",
    "num_classified_rois",
    "mean_video_conf",
    "mean_still_conf",
    "mean_conf_gain",
    "median_conf_gain",
    "positive_conf_gain_rate",
    "pred_change_rate",
    "mean_entropy_gain",
    "small_object_mean_conf_gain",
    "conf_gain_per_kb",
    "positive_gain_events_per_mb",
]

plot_df = summary_df[[c for c in main_cols if c in summary_df.columns]].copy()

policy_order = ["permissive", "balanced_top2", "conf_size_top1", "strict_small_only"]
plot_df["policy_name"] = pd.Categorical(plot_df["policy_name"], categories=policy_order, ordered=True)
plot_df = plot_df.sort_values(["sequence_name", "regime", "policy_name"])

plot_df.head()

## 7. Aggregate across sequences by regime and policy

In [ ]:
agg_df = (
    plot_df.groupby(["regime", "policy_name"], observed=True)
    .agg(
        num_sequences=("sequence_name", "nunique"),
        mean_num_selected_rois=("num_selected_rois", "mean"),
        mean_roi_selection_ratio=("roi_selection_ratio", "mean"),
        mean_frame_roi_coverage=("frame_roi_coverage", "mean"),
        mean_roi_bitrate_mbps=("roi_bitrate_mbps", "mean"),
        mean_roi_bitrate_share=("roi_bitrate_share", "mean"),
        mean_mean_conf_gain=("mean_conf_gain", "mean"),
        mean_median_conf_gain=("median_conf_gain", "mean"),
        mean_positive_conf_gain_rate=("positive_conf_gain_rate", "mean"),
        mean_mean_entropy_gain=("mean_entropy_gain", "mean"),
        mean_conf_gain_per_kb=("conf_gain_per_kb", "mean"),
    )
    .reset_index()
)

agg_df

In [ ]:
agg_csv = OUT_DIR / "aggregated_policy_summary.csv"
agg_df.to_csv(agg_csv, index=False)
print("Saved:", agg_csv)

## 8. Plot 1 — Mean confidence gain vs ROI bitrate

In [ ]:
plt.figure(figsize=(8, 6))

for regime in sorted(agg_df["regime"].dropna().unique()):
    sub = agg_df[agg_df["regime"] == regime].copy()
    plt.scatter(
        sub["mean_roi_bitrate_mbps"],
        sub["mean_mean_conf_gain"],
        s=80,
        label=f"{regime}",
    )
    for _, row in sub.iterrows():
        plt.annotate(
            str(row["policy_name"]),
            (row["mean_roi_bitrate_mbps"], row["mean_mean_conf_gain"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
        )

plt.xlabel("ROI bitrate (Mbps)")
plt.ylabel("Mean confidence gain")
plt.title("Semantic gain vs ROI bitrate")
plt.axhline(0.0, linewidth=1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plot1 = OUT_DIR / "plot_conf_gain_vs_roi_bitrate.png"
plt.savefig(plot1, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", plot1)

## 9. Plot 2 — Mean confidence gain vs ROI bitrate share

In [ ]:
plt.figure(figsize=(8, 6))

for regime in sorted(agg_df["regime"].dropna().unique()):
    sub = agg_df[agg_df["regime"] == regime].copy()
    plt.scatter(
        sub["mean_roi_bitrate_share"],
        sub["mean_mean_conf_gain"],
        s=80,
        label=f"{regime}",
    )
    for _, row in sub.iterrows():
        plt.annotate(
            str(row["policy_name"]),
            (row["mean_roi_bitrate_share"], row["mean_mean_conf_gain"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
        )

plt.xlabel("ROI bitrate share")
plt.ylabel("Mean confidence gain")
plt.title("Semantic gain vs ROI bitrate share")
plt.axhline(0.0, linewidth=1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plot2 = OUT_DIR / "plot_conf_gain_vs_roi_share.png"
plt.savefig(plot2, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", plot2)

## 10. Plot 3 — Mean selected ROIs by policy

In [ ]:
for regime in sorted(agg_df["regime"].dropna().unique()):
    sub = agg_df[agg_df["regime"] == regime].copy()

    plt.figure(figsize=(8, 5))
    plt.bar(sub["policy_name"].astype(str), sub["mean_num_selected_rois"])
    plt.xlabel("Policy")
    plt.ylabel("Mean selected ROIs")
    plt.title(f"Selected ROIs by policy ({regime})")
    plt.xticks(rotation=20)
    plt.tight_layout()

    out = OUT_DIR / f"plot_selected_rois_{regime}.png"
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

## 11. Plot 4 — Positive confidence gain rate by policy

In [ ]:
for regime in sorted(agg_df["regime"].dropna().unique()):
    sub = agg_df[agg_df["regime"] == regime].copy()

    plt.figure(figsize=(8, 5))
    plt.bar(sub["policy_name"].astype(str), sub["mean_positive_conf_gain_rate"])
    plt.xlabel("Policy")
    plt.ylabel("Positive confidence gain rate")
    plt.title(f"Positive confidence gain rate by policy ({regime})")
    plt.xticks(rotation=20)
    plt.ylim(0, 1)
    plt.tight_layout()

    out = OUT_DIR / f"plot_positive_conf_gain_rate_{regime}.png"
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

## 12. Plot 5 — Confidence gain per KB by policy

In [ ]:
for regime in sorted(agg_df["regime"].dropna().unique()):
    sub = agg_df[agg_df["regime"] == regime].copy()

    plt.figure(figsize=(8, 5))
    plt.bar(sub["policy_name"].astype(str), sub["mean_conf_gain_per_kb"])
    plt.xlabel("Policy")
    plt.ylabel("Confidence gain per KB")
    plt.title(f"Utility efficiency by policy ({regime})")
    plt.xticks(rotation=20)
    plt.axhline(0.0, linewidth=1)
    plt.tight_layout()

    out = OUT_DIR / f"plot_conf_gain_per_kb_{regime}.png"
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

## 13. Plot 6 — Mean entropy gain vs ROI bitrate

In [ ]:
plt.figure(figsize=(8, 6))

for regime in sorted(agg_df["regime"].dropna().unique()):
    sub = agg_df[agg_df["regime"] == regime].copy()
    plt.scatter(
        sub["mean_roi_bitrate_mbps"],
        sub["mean_mean_entropy_gain"],
        s=80,
        label=f"{regime}",
    )
    for _, row in sub.iterrows():
        plt.annotate(
            str(row["policy_name"]),
            (row["mean_roi_bitrate_mbps"], row["mean_mean_entropy_gain"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
        )

plt.xlabel("ROI bitrate (Mbps)")
plt.ylabel("Mean entropy gain")
plt.title("Entropy reduction vs ROI bitrate")
plt.axhline(0.0, linewidth=1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plot6 = OUT_DIR / "plot_entropy_gain_vs_roi_bitrate.png"
plt.savefig(plot6, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", plot6)

## 14. Save paper-oriented comparison table

In [ ]:
paper_table = agg_df.copy()
paper_table = paper_table.rename(columns={
    "mean_num_selected_rois": "selected_rois",
    "mean_roi_selection_ratio": "roi_selection_ratio",
    "mean_frame_roi_coverage": "frame_roi_coverage",
    "mean_roi_bitrate_mbps": "roi_bitrate_mbps",
    "mean_roi_bitrate_share": "roi_bitrate_share",
    "mean_mean_conf_gain": "mean_conf_gain",
    "mean_median_conf_gain": "median_conf_gain",
    "mean_positive_conf_gain_rate": "positive_conf_gain_rate",
    "mean_mean_entropy_gain": "mean_entropy_gain",
    "mean_conf_gain_per_kb": "conf_gain_per_kb",
})

paper_table_csv = OUT_DIR / "paper_policy_comparison_table.csv"
paper_table.to_csv(paper_table_csv, index=False)

print("Saved:", paper_table_csv)
paper_table

## 15. Optional per-sequence scatter for the low regime

In [ ]:
low_df = plot_df[plot_df["regime"] == "low"].copy()

if len(low_df) == 0:
    print("No low-regime summaries available.")
else:
    plt.figure(figsize=(8, 6))
    for policy in policy_order:
        sub = low_df[low_df["policy_name"] == policy]
        if len(sub) == 0:
            continue
        plt.scatter(
            sub["roi_bitrate_mbps"],
            sub["mean_conf_gain"],
            s=60,
            label=str(policy),
        )

    plt.xlabel("ROI bitrate (Mbps)")
    plt.ylabel("Mean confidence gain")
    plt.title("Per-sequence tradeoff (low regime)")
    plt.axhline(0.0, linewidth=1)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out = OUT_DIR / "plot_per_sequence_low_tradeoff.png"
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", out)

## 16. Summary of generated files

In [ ]:
generated = sorted(OUT_DIR.glob("*"))
print(f"Generated {len(generated)} files:")
for p in generated:
    print("-", p.name)

## 17. Suggested figures to include in the paper

The most useful first figures are:
- `plot_conf_gain_vs_roi_bitrate.png`
- `plot_conf_gain_vs_roi_share.png`
- `plot_selected_rois_low.png`
- `plot_positive_conf_gain_rate_low.png`
- `plot_conf_gain_per_kb_low.png`

These are the clearest way to show the bitrate–semantic-gain tradeoff across policies.